In [ ]:
#前面chromaDemo的代码导入
import os
from openai import OpenAI
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv

# from langchain.retrievers.document_compressors import CrossEncoderReranker
# from langchain.retrievers import ContextualCompressedRetriever


load_dotenv()

def get_metadata_str(metadata):
    if metadata is None:
        return ""
    _str=""
    for key, value in metadata.items():
        _str += f"{key}: {value}\n"
    return _str

metadata ={
    '标题':'科技行业2025年展望',
    '作者':'沈岱，马智焱，黄佳琦',
    '发表时间':"2024年12月13日"
}

from langchain_community.embeddings import DashScopeEmbeddings
embeddings=DashScopeEmbeddings(model='text-embedding-v4',
dashscope_api_key=os.environ['DASHSCOPE_API_KEY'])

from langchain_community.document_loaders import TextLoader
loader = TextLoader('/root/ai_rag_project/data_base/科技行业 2025 年展望.txt')
docs = loader.load()
for doc in docs:
    doc.metadata.update(metadata)

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=100)
all_splitters = text_splitter.split_documents(docs)

from langchain_chroma import Chroma
vector_store = Chroma(embedding_function=embeddings)
batch_size = 10
all_ids =[]
for i in range(0,len(all_splitters),batch_size):
    batch=all_splitters[i:i+batch_size]
    cur_ids=vector_store.add_documents(batch)
    all_ids.extend(cur_ids)
ids = all_ids


from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {query} 
Context: {context} 
Answer:""")

llm=ChatOpenAI(api_key=os.environ['DASHSCOPE_API_KEY'],
               base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
               model="qwen3.6-plus"
               )
retriever=vector_store.as_retriever()
# def query_vector(info):
#     # retriever=vector_store.as_retriever(search_kwargs={"k":10})
#     docs = retriever.invoke(info["query"])
#     formatted_docs = []
#     for doc in docs:
#         meta = get_metadata_str(doc.metadata)
#         formatted_docs.append(f"【参考资料】\n{meta}【内容】\n{doc.page_content}")
        
#     return "\n\n".join(formatted_docs)

# output_parser = StrOutputParser()

# rag_chain = ( {"context":query_vector, "query": lambda x: x["query"]}| prompt | llm | output_parser)

# result = rag_chain.invoke({'query':"2025年AI服务器出货量预计是多少"})


使用langchain的CrossEncoder进行重排

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors.base import BaseDocumentCompressor
from typing import Optional, Sequence
from dashscope import TextReRank
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun

# 1. 封装类：定义阿里云重排逻辑
class AliyunRerank(BaseDocumentCompressor):
    model_name: str = "gte-rerank-v2"
    top_n: int = 3

    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Optional[CallbackManagerForRetrieverRun] = None,
    ) -> Sequence[Document]:
        if not documents:
            return []
        
        doc_contents = [doc.page_content for doc in documents]
        resp = TextReRank.call(
            model=self.model_name,
            query=query,
            documents=doc_contents,
            top_n=self.top_n,
            api_key=os.environ.get('DASHSCOPE_API_KEY')
        )

        if resp.status_code != 200:
            print(f"Rerank 失败: {resp.message}")
            return documents[:self.top_n]

        final_results = []
        for res in resp.output.results:
            doc = documents[res.index] # 关键：找回原始带 metadata 的文档
            doc.metadata["rerank_score"] = res.relevance_score
            final_results.append(doc)
        return final_results

# 2. 实例化并构建新的检索器
# 注意：这里我们让初筛 (k=15) 找回更多内容，给重排器精挑细选的空间
ali_compressor = AliyunRerank(top_n=3)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 15})

# 这是“升级版”的检索器
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=ali_compressor, 
    base_retriever=base_retriever
)

def query_vector(info):
    # 改用这个带重排功能的检索器
    docs = rerank_retriever.invoke(info["query"])
    
    formatted_docs = []
    for doc in docs:
        meta = get_metadata_str(doc.metadata)
        formatted_docs.append(f"【参考资料】\n{meta}【内容】\n{doc.page_content}")
        
    return "\n\n".join(formatted_docs)



In [ ]:
output_parser = StrOutputParser()

rag_chain = ( {"context":query_vector, "query": lambda x: x["query"]}| prompt | llm | output_parser)
result=rag_chain.invoke({'query':'2024年中东非智能手机的出货量为多少？'})
result